# 04 — Transformer (DistilBERT) on Colab

**Goal:** fine-tune `distilbert-base-uncased` on the train split, export it to ONNX (int8),
check it fits the deployment limits, and hand the model to DVC so `dvc repro` can evaluate it
next to the classical models.

**How it is set up:**
- This notebook runs the `train_transformer` DVC stage's command
  (`python -m fjd.models.train_transformer`). All the logic lives in `src/fjd/models/`, so
  Colab and the pipeline train the same way. Settings are in `params.yaml: transformer`.
- The input is one text per posting: title, short fields, the binary flags as words
  (`company logo: no`), then the long fields, each cut to a word budget, up to 256 tokens
  (`fjd.models.transformer.posting_text`).
- The best epoch is picked by val PR-AUC. The threshold and the reported metrics come from the
  **exported int8 model**, because that is what gets deployed.
- The stage is `frozen` in `dvc.yaml`, so `dvc repro` never retrains it (not even with `-f`). After
  a run here, the model is recorded with `dvc commit` (section 5).
- GPU training isn't bit-for-bit reproducible. A retrained model has to match the committed val
  PR-AUC within `transformer.repro_tolerance` (0.005); section 4 checks this.

**Run it on Colab:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Add two Colab secrets (key icon in the sidebar): `DAGSHUB_USER` and `DAGSHUB_TOKEN`. A free
   DagsHub account's token is enough (read access, to `dvc pull` the data).
3. Set `BRANCH` below and *Run all*. Training takes roughly 10 minutes on a T4.

You can also run it locally from the repo root, if you have a GPU and ran
`uv sync --group transformer`.

In [ ]:
REPO_URL = "https://github.com/webmaster99-stack/fake_job_posting_detector.git"
BRANCH = "main"  # the branch whose code and params.yaml to train with

## 1. Set up

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def sh(*cmd: str) -> None:
    """Run a command, streaming its output into the notebook (a child process's own output
    would go to the kernel's log instead)."""
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


if IN_COLAB:
    from google.colab import userdata

    ROOT = Path("/content/fake_job_posting_detector")
    if not ROOT.exists():
        sh("git", "clone", "-b", BRANCH, REPO_URL, str(ROOT))
    os.chdir(ROOT)
    # uv installs the exact versions from uv.lock, into .venv (not Colab's own Python).
    sh("pip", "install", "-q", "uv")
    sh("uv", "sync", "-q", "--no-default-groups", "--group", "transformer")
    PY = str(ROOT / ".venv" / "bin" / "python")
    for key in ("DAGSHUB_USER", "DAGSHUB_TOKEN"):
        os.environ[key] = userdata.get(key)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(ROOT)
    PY = sys.executable  # the project's uv environment


def run(*args: str) -> None:
    """Run the project's Python from the repo root."""
    sh(PY, *args)


run("-m", "fjd.utils.dvc_auth")
run("-m", "dvc", "pull", "split")  # only the processed train/val/test splits
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Training on a CPU works but takes hours.
run("-c", "import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name())")

## 2. Fine-tune and export

Writes `models/transformer/` (`model.onnx`, `tokenizer.json`, `text.json`, `threshold.json`) and
`reports/train/transformer.json`. The pretrained checkpoint is pinned to a Hub commit
(`transformer.revision`).

In [ ]:
run("-m", "fjd.models.train_transformer")

## 3. Results and deployability

In [ ]:
import pandas as pd
import yaml

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))
report = json.loads(Path("reports/train/transformer.json").read_text(encoding="utf-8"))

print("Per epoch (torch model, val split):")
display(pd.DataFrame(report["history"]).T)

val = report["val"]
print(f"best epoch, torch fp32:  val PR-AUC {report['best_torch_val_pr_auc']:.4f}")
kind = "ONNX int8" if params["transformer"]["quantize"] else "ONNX fp32"
print(f"exported ({kind}):     val PR-AUC {val['pr_auc']:.4f}")
print(
    f"at the F1-optimal threshold: precision {val['precision']:.3f}, "
    f"recall {val['recall']:.3f}, F1 {val['f1']:.3f}"
)

The deployability check times single postings on one CPU thread, the way the API will
serve them. These numbers come from Colab's CPU. `dvc repro` measures them again on your machine
and logs them to MLflow. The limits are from `params.yaml: selection`.

In [ ]:
limits = params["selection"]
d = report["deployability"]
print(f"size:        {d['size_mb']:.1f} MB   (limit {limits['max_model_size_mb']} MB)")
print(
    f"p95 latency: {d['p95_latency_ms']:.1f} ms  "
    f"(limit {limits['max_p95_latency_ms']} ms; p50 {d['p50_latency_ms']:.1f} ms)"
)
print("deployable" if d["deployable"] else "NOT deployable: over a limit")

## 4. Reproduction check

Compares this run with the model committed on `BRANCH`, if there is one.

In [ ]:
committed = subprocess.run(
    ["git", "show", "HEAD:reports/train/transformer.json"], capture_output=True, text=True
)
if committed.returncode != 0:
    print("No committed transformer yet: this run becomes the reference.")
else:
    ref = json.loads(committed.stdout)["val"]["pr_auc"]
    delta = abs(val["pr_auc"] - ref)
    tolerance = params["transformer"]["repro_tolerance"]
    verdict = "reproduced" if delta <= tolerance else "NOT reproduced"
    print(f"val PR-AUC {val['pr_auc']:.4f} vs committed {ref:.4f}: |Δ| = {delta:.4f}")
    print(f"{verdict} (tolerance {tolerance})")

## 5. Record the model in DVC (maintainer)

Download the outputs:

In [ ]:
import zipfile

archive = Path("transformer_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in [*Path("models/transformer").iterdir(), Path("reports/train/transformer.json")]:
        z.write(path, path.as_posix())
print(f"{archive} ({archive.stat().st_size / 1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(archive))

Then, in your local clone, on the same commit as `BRANCH`:

```bash
unzip -o transformer_outputs.zip    # from the repo root: models/transformer/ + the report
uv run dvc commit -f train_transformer   # record the outputs in dvc.lock (the stage is frozen)
uv run dvc repro                    # `evaluate` scores the new model on val and test
uv run dvc push                     # upload the model to DagsHub (needs write access)
git add dvc.lock reports/ && git commit -m "Retrain transformer"
```

`dvc commit` hashes the stage's dependencies on your machine. They are the same files Colab
trained on, since both checkouts are the same commit with the same `dvc pull` data.